<a href="https://colab.research.google.com/github/Dima200206/-2/blob/main/%D0%9B%D0%914_%D0%92%D0%BB%D0%B0%D1%81%D0%B5%D0%BD%D0%BA%D0%BE%20%D0%94%D0%BC%D0%B8%D1%82%D1%80%D0%BE%20%D0%86%D0%BD%D1%82%D0%B5%D0%BB%D0%B5%D0%BA%D1%82%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D1%96%20%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8%20%D0%A4%D0%86%D0%A2%201-4%20%D0%BC%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Лабораторна 4

In [1]:
# Крок 1: Встановлення необхідного середовища
!pip install pgmpy -q

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Крок 2: Опис архітектури графа (DAG)
# Використовуємо DiscreteBayesianNetwork для сумісності з новими версіями pgmpy
movie_model = DiscreteBayesianNetwork([
    ('User_Mood', 'Genre_Preference'),
    ('Genre_Preference', 'Movie_Recommendation'),
    ('Age_Group', 'Movie_Recommendation'),
    ('Movie_Recommendation', 'User_Satisfaction')
])

# Крок 3: Створення таблиць умовних ймовірностей (CPT)

# Настрій: Happy (0.4), Sad (0.3), Exciting (0.3)
cpd_mood = TabularCPD(variable='User_Mood', variable_card=3,
                      values=[[0.4], [0.3], [0.3]],
                      state_names={'User_Mood': ['Happy', 'Sad', 'Exciting']})

# Вік: Adult (0.8), Child (0.2)
cpd_age = TabularCPD(variable='Age_Group', variable_card=2,
                     values=[[0.8], [0.2]],
                     state_names={'Age_Group': ['Adult', 'Child']})

# Жанр залежить від настрою: Comedy, Drama, Action
cpd_genre = TabularCPD(variable='Genre_Preference', variable_card=3,
    values=[
        [0.7, 0.1, 0.2], # Ймовірність Comedy
        [0.1, 0.8, 0.1], # Ймовірність Drama
        [0.2, 0.1, 0.7]  # Ймовірність Action
    ],
    evidence=['User_Mood'], evidence_card=[3],
    state_names={'Genre_Preference': ['Comedy', 'Drama', 'Action'],
                 'User_Mood': ['Happy', 'Sad', 'Exciting']})

# Рекомендація залежить від Жанру та Віку
# Стани: Comedy_Movie, Drama_Movie, Action_Movie, Cartoon
cpd_recom = TabularCPD(variable='Movie_Recommendation', variable_card=4,
    values=[
        [0.9, 0.05, 0.05, 0.2, 0.1, 0.1], # Comedy_Movie
        [0.05, 0.9, 0.05, 0.1, 0.2, 0.1], # Drama_Movie
        [0.05, 0.05, 0.9, 0.1, 0.1, 0.2], # Action_Movie
        [0.0, 0.0, 0.0, 0.6, 0.6, 0.6]    # Cartoon
    ],
    evidence=['Genre_Preference', 'Age_Group'], evidence_card=[3, 2],
    state_names={'Movie_Recommendation': ['Comedy_Movie', 'Drama_Movie', 'Action_Movie', 'Cartoon'],
                 'Genre_Preference': ['Comedy', 'Drama', 'Action'],
                 'Age_Group': ['Adult', 'Child']})

# Задоволеність користувача: High, Low
cpd_satisfaction = TabularCPD(variable='User_Satisfaction', variable_card=2,
    values=[[0.9, 0.8, 0.85, 0.95], [0.1, 0.2, 0.15, 0.05]],
    evidence=['Movie_Recommendation'], evidence_card=[4],
    state_names={'User_Satisfaction': ['High', 'Low'],
                 'Movie_Recommendation': ['Comedy_Movie', 'Drama_Movie', 'Action_Movie', 'Cartoon']})

# Додавання параметрів та перевірка моделі
movie_model.add_cpds(cpd_mood, cpd_age, cpd_genre, cpd_recom, cpd_satisfaction)
print(f"Модель коректна: {movie_model.check_model()}")

inference = VariableElimination(movie_model)

# 4. Прогнозування (Прямий вивід)
print("\n--- ПРОГНОЗУВАННЯ ---")
# Сценарій: Користувач дорослий, настрій сумний (Sad). Яка ймовірність драми?
res1 = inference.query(variables=['Movie_Recommendation'], evidence={'Age_Group': 'Adult', 'User_Mood': 'Sad'})
print(res1)

# Сценарій: Дитина, настрій веселий (Happy).
res2 = inference.query(variables=['Movie_Recommendation'], evidence={'Age_Group': 'Child', 'User_Mood': 'Happy'})
print(res2)

# 5. Діагностика (Зворотний вивід)
print("\n--- ДІАГНОСТИКА ---")
# Сценарій: Користувач незадоволений (Low satisfaction). Який настрій міг бути причиною?
diag1 = inference.query(variables=['User_Mood'], evidence={'User_Satisfaction': 'Low'})
print(diag1)

# Сценарій: Система видала рекомендацію Cartoon. Яка ймовірність, що користувач — дитина?
diag2 = inference.query(variables=['Age_Group'], evidence={'Movie_Recommendation': 'Cartoon'})
print(diag2)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 33.7 MB/s eta 0:00:00
Модель коректна: True

--- ПРОГНОЗУВАННЯ ---
+------------------------------------+-----------------------------+
| Movie_Recommendation               |   phi(Movie_Recommendation) |
+====================================+=============================+
| Movie_Recommendation(Comedy_Movie) |                      0.1400 |
+------------------------------------+-----------------------------+
| Movie_Recommendation(Drama_Movie)  |                      0.0650 |
+------------------------------------+-----------------------------+
| Movie_Recommendation(Action_Movie) |                      0.7350 |
+------------------------------------+-----------------------------+
| Movie_Recommendation(Cartoon)      |                      0.0600 |
+------------------------------------+-----------------------------+
+------------------------------------+